# Synthetic alpha-signal benchmark

Reproducible research prototype, November 2026. Developed with AI assistance. **All data and returns are simulated.** The result snapshot comes from the executed companion script; rerun the cells to reproduce the experiment.

## Hypothesis and validation

When predictability is weak and observations are serially dependent, does a nonlinear model improve one-period-ahead return forecasting over a regularized linear baseline, and does the forecasting edge survive stylized transaction costs?

## Data and timing

All 12,000 observations are simulated locally with seed 42. Eight observable AR(1) processes have persistence from 0.25 to 0.85. The next-period return has a deliberately planted conditional mean and Gaussian noise. Noise volatility rises after observation 6,000. A ninth feature is the previous observed return.

At decision index t, the observable vector and return through t are known; the target is the return of period t+1. There are no prices, real assets or downloaded market data. The planted relationship makes this an implementation and evaluation exercise, not an alpha discovery claim.

## Protocol

- Initial development period: indices 0-3,999. Hyperparameters use train indices 0-2,998 and validation 3,000-3,999; index 2,999 is excluded.
- Ridge candidates: alpha 0.1, 1, 10 and 100. Random Forest candidates: depth 3/6, minimum leaf size 25/75, 100 trees, seed 42.
- Hyperparameter selection happens once, before outer testing. Random Forest wins initial validation MSE; both fitted model families are still reported on every outer test.
- Four chronological expanding-window tests: 4,000-5,999; 6,000-7,999; 8,000-9,999; 10,000-11,999. One row separates training and each outer test. Training expands; each test point is predicted once per model.
- Ridge scaling is fitted only on each training window. No feature selection or retuning uses outer-test results.
- Report pooled/per-fold MSE, MAE, Pearson information coefficient (IC), Spearman rank IC and a descriptive 95% circular block-bootstrap interval for pooled IC (block length 20; 1,000 draws).
- The zero-return forecast is a forecasting baseline, not an investable benchmark. Its IC is undefined, since its prediction is constant.



In [ ]:
from run_benchmark import main
main()


In [ ]:
import json
from pathlib import Path
import pandas as pd
results = json.loads(Path('results.json').read_text())
pd.DataFrame(results['metrics'])


## Verified result snapshot

| Model | OOS MSE | Pearson IC | Spearman IC | MSE reduction vs zero |
|---|---:|---:|---:|---:|
| ZeroForecast | 0.00012604 | n/a | n/a | 0.000% |
| Ridge | 0.00012495 | 0.0947 | 0.0967 | 0.864% |
| RandomForest | 0.00012493 | 0.0953 | 0.0950 | 0.876% |

Random Forest is almost indistinguishable from Ridge in forecast error. Under stylized 10 bps turnover costs, its simulated annualized Sharpe falls to 0.157. These are simulation results, not market alpha.

In [ ]:
pd.DataFrame(results['fold_metrics'])


In [ ]:
pd.DataFrame(results['cost_sensitivity'])


## Limits and research judgment

A known planted relationship permits controlled testing but says nothing about actual trading opportunities. Point-in-time data, execution realism and multiple-testing controls are required for market research. See README.md for details.